# L12 — Project 5: Group Synchrony

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L12/05_group_synchrony/synchrony_starter.ipynb)

**Goal:** measure how synchronized two dancers are across a routine. Two motion traces, one synchrony score over time, plus a cross-correlation that says *who's leading whom*.

**Why a single number isn't enough.** Two dancers might be locked-in for the chorus and drift on the bridge. A score *over time* tells the coach where to focus. The bundled recordings show this on purpose: dancer B lags dancer A by 150 ms during the middle 12 seconds — windowed correlation will drop in that segment.

> **New to pandas / numpy / scipy?** Skim [`L12/00_python_data_tools/python_data_tools_starter.ipynb`](../00_python_data_tools/python_data_tools_starter.ipynb) first — it's a 30-minute tour of every function this notebook uses, with tiny standalone examples.

## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Step 1 — Load both recordings

Each dancer wore a wrist-mounted micro:bit running the L11 datalogger program. The recordings are time-aligned — both started at the same `t=0`.

In [ ]:
URL_A = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-pair-A.csv"
URL_B = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-pair-B.csv"

if IN_COLAB:
    a = pd.read_csv(URL_A)
    b = pd.read_csv(URL_B)
else:
    a = pd.read_csv("../../data/sample-pair-A.csv")
    b = pd.read_csv("../../data/sample-pair-B.csv")

for d in (a, b):
    d["mag"] = np.sqrt(d["x"]**2 + d["y"]**2 + d["z"]**2)

HZ = 20
print("A:", a.shape, "   B:", b.shape)

> **Loading your *own* paired captures later?** Put both CSVs in Google Drive, mount Drive in Colab, then load with `pd.read_csv('/content/drive/MyDrive/.../dancer-A.csv')` (and B). The *Working in Colab* section of [`L12/00_python_data_tools/`](../00_python_data_tools/python_data_tools_starter.ipynb) covers the mount and explains why files dragged into the Files panel disappear when the runtime times out.

## Step 2 — Plot the two traces on the same axes

Eyeball test first. They should hug each other most of the time, with the middle segment looking visibly off.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(a["t"], a["mag"], color="steelblue", linewidth=0.7, label="dancer A")
ax.plot(b["t"], b["mag"], color="crimson",   linewidth=0.7, label="dancer B")
ax.axvspan(6, 18, color="yellow", alpha=0.15, label="expected drift")
ax.set_xlabel("time (s)"); ax.set_ylabel("|a| (mg)"); ax.legend()
plt.show()

## Step 3 — One-number synchrony: Pearson correlation

**Pearson correlation** is a number from –1 to +1 that says how much two signals move together. +1 = perfectly together, 0 = unrelated, –1 = perfectly opposite.

In [ ]:
rho = np.corrcoef(a["mag"], b["mag"])[0, 1]
print(f"overall Pearson correlation: {rho:.3f}")

## Step 4 — A score *over time*: windowed correlation

The single number averages out where the dancers were synced and where they weren't. Slide a 1-second window across both traces, compute Pearson correlation in each window, plot the result.

The drop in the highlighted region is the lesson — that's where dancer B is lagging.

In [ ]:
WINDOW_S = 1.0
W = int(WINDOW_S * HZ)
centers, scores = [], []
for start in range(0, min(len(a), len(b)) - W, W // 2):
    a_win = a["mag"].iloc[start:start + W].to_numpy()
    b_win = b["mag"].iloc[start:start + W].to_numpy()
    if a_win.std() > 1 and b_win.std() > 1:        # don't correlate constants
        scores.append(np.corrcoef(a_win, b_win)[0, 1])
    else:
        scores.append(np.nan)
    centers.append(a["t"].iloc[start + W // 2])

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(centers, scores, "o-", color="purple")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvspan(6, 18, color="yellow", alpha=0.15)
ax.set_xlabel("time (s)"); ax.set_ylabel("windowed correlation")
ax.set_ylim(-1, 1)
plt.show()

## Step 5 — Cross-correlation: who's leading?

**Cross-correlation** asks: "if I shift one signal by `lag` samples, how much does it line up with the other?" The lag with the highest correlation is the time offset between the dancers. Negative = A leads, positive = B leads.

In [ ]:
# Look at just the middle drift segment (t = 6..18 s).
mid = (a["t"] >= 6) & (a["t"] < 18)
x = a["mag"][mid].to_numpy() - a["mag"][mid].mean()
y = b["mag"][mid].to_numpy() - b["mag"][mid].mean()

xc = correlate(x, y, mode="full")
lags = np.arange(-len(x) + 1, len(x)) / HZ           # in seconds
best_lag = lags[np.argmax(xc)]
print(f"best lag in middle segment: {best_lag * 1000:+.0f} ms")
print("(negative = dancer B is delayed relative to dancer A)")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lags, xc, color="crimson")
ax.axvline(best_lag, color="black", linestyle="--", label=f"best lag = {best_lag*1000:+.0f} ms")
ax.set_xlim(-1, 1)
ax.set_xlabel("lag (s)")
ax.set_ylabel("cross-correlation")
ax.legend()
plt.show()

## What to build next (L13 starting goals)

1. **Capture paired data.** Two students wear the L11 datalogger rig and perform the same routine simultaneously. Either start both with a clap (visible in the magnitude trace; you can use it to align timestamps post-hoc) or sync clocks first.
2. **Three or more dancers.** Average all pairwise correlations → ensemble synchrony score. Which dancer is the outlier?
3. **Live lantern feedback.** Send the windowed score to the lanterns: green when synced (>0.7), yellow when drifting (0.3–0.7), red when way off (<0.3).
4. **Per-move synchrony.** Combine with the L11 labeling pattern — which moves does the ensemble reliably nail, and which fall apart?
5. **Rehearsal scorecard.** Log the per-session ensemble score across weeks. Is the group getting tighter?

## Reflect (homework)

Write 3–4 sentences:
- Which extension will you build first?
- How will you handle clock synchronization between the two micro:bits when capturing?
- What would Pearson correlation = 0 look like, and why is it different from negative correlation?